## Setup

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path
)

documents = [file.parse() for file in reader.read()]
print(f"The total of documents is {len(documents)}")

The total of documents is 72


## Generating ground truth

In [2]:
import os
import json
from dotenv import load_dotenv
from pydantic import BaseModel
from openai import OpenAI
from evaluation_utils import llm_structured_retry

load_dotenv()

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """\
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

openai_client = OpenAI(
    base_url="https://ai.sumopod.com/v1",
    api_key=os.getenv("SUMOPOD_API_KEY")
)

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    
    out, usage = llm_structured_retry(
        user_prompt=user_prompt,
        client=openai_client,
        instructions=data_gen_instructions,
        model="gpt-5.4-mini",
        max_retries=3,
        output_type=Questions
    )
    
    results = []
    for question in out.questions:
        results.append({
            "document": doc["filename"],
            "question": question
        })

    return results, usage

print("Ground truth generator test:")
generate_ground_truth(documents[0])

Ground truth generator test:


([{'document': '01-agentic-rag/lessons/01-intro.md',
   'question': 'What problem does retrieval-augmented generation solve when you want an LLM to answer from your own documents?'},
  {'document': '01-agentic-rag/lessons/01-intro.md',
   'question': 'Why does this course build the RAG system in plain Python instead of using a framework right away?'},
  {'document': '01-agentic-rag/lessons/01-intro.md',
   'question': 'What are the main limits of an LLM that make RAG useful in the first place?'},
  {'document': '01-agentic-rag/lessons/01-intro.md',
   'question': 'What kind of demo project are we building in this module to show RAG working end to end?'},
  {'document': '01-agentic-rag/lessons/01-intro.md',
   'question': 'What will be covered in the first part of the module, and how is the second part different?'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0, audio_tokens=None, text_tokens=None), output_tokens=117, output_tokens_details=Ou

## Q1. Generating questions

In [3]:
sample_filenames = [
    "01-agentic-rag/lessons/01-intro.md", 
    "01-agentic-rag/lessons/02-environment.md", 
    "01-agentic-rag/lessons/03-rag.md"
]

all_usages = []
all_results = []

for filename in sample_filenames:
    for doc in documents:
        if doc["filename"] == filename:
            results, usage = generate_ground_truth(doc)
            all_usages.append(usage)
            all_results.extend(results)

total_input_tokens = sum(usage.input_tokens for usage in all_usages)
avg_input_tokens = total_input_tokens / len(all_usages)

print(f"The average input tokens of the 3 examples is {avg_input_tokens}")

The average input tokens of the 3 examples is 1353.0


## The full ground truth

In [4]:
import pandas as pd
from tqdm.contrib.concurrent import thread_map

ground_truth = thread_map(
    generate_ground_truth, 
    documents, 
    max_workers=6
)

records = []
usages = []

for record, usage in ground_truth:
    records.extend(record)
    usages.append(usage)

df_records = pd.DataFrame(records)
df_records.to_csv("ground-truth.csv", index=False)

  0%|          | 0/72 [00:00<?, ?it/s]

In [5]:
import pandas as pd

df_records = pd.read_csv("ground-truth.csv")
records = df_records.to_dict(orient="records")

## Searching the chunks

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [7]:
import numpy as np
from fastembed import TextEmbedding

chunks_content = [chunk["content"] for chunk in chunks]

embedder_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
embeddings = np.array(list(embedder_model.embed(chunks_content)))

In [8]:
from minsearch import VectorSearch, Index

index = Index(text_fields=["content"])
vindex = VectorSearch(keyword_fields=["content"])

index.fit(chunks)
vindex.fit(embeddings, chunks)

In [9]:
def text_search(query, num_results=5):
    results = index.search(query=query, num_results=num_results)
    return results

def vector_search(query, embedder=embedder_model, num_results=5):
    vector = np.array(list(embedder.embed(query)))
    vector_flatten = vector.flatten()
    results = vindex.search(vector_flatten, num_results=num_results)
    return results

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10, embedder=embedder_model)
    return rrf([text_results, vector_results], k=k)

## Q2. First result with text search

In [10]:
q = records[0]["question"]

print("First result of text search:")
text_search(q)[0]

First result of text search:


{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

## Q3. First result with vector search

In [11]:
print("First result of vector search:")
vector_search(q, embedder=embedder_model)[0]

First result of vector search:


{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

## Evaluation metrics

In [12]:
def compute_relevance(record, search_function):
    label = record["document"]
    results = search_function(record["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == label))

    return relevance

def compute_relevance_total(records, search_function):
    relevance_total = []

    for record in records:
        relevance = compute_relevance(record, search_function)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevances):
    hit = 0
    for row in relevances:
        if 1 in row:
            hit += 1
            
    score = hit / len(relevances)
    return score

def mrr(relevances):
    total_score = 0.0
    for line in relevances:
        for rank, value in enumerate(line):
            if value == 1:
                total_score += 1 / (rank + 1)
                break
                
    return total_score / len(relevances)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }    

## Q4. Evaluating text search

In [13]:
print("The metric results:")
evaluate(records, text_search)

The metric results:


{'hit_rate': 0.8111111111111111, 'mrr': 0.6361574074074076}

## Q5. Evaluating vector search

In [14]:
print("The metric results:")
evaluate(records, vector_search)

The metric results:


{'hit_rate': 0.7527777777777778, 'mrr': 0.5485648148148147}

## Q6. Tuning hybrid search

In [15]:
for k in [1, 50, 100, 200]:
    result = evaluate(
        records,
        lambda query, k=k: hybrid_search(query, k) 
    )
    print(f"k={k}: {result}")

k=1: {'hit_rate': 0.875, 'mrr': 0.6739351851851851}
k=50: {'hit_rate': 0.8722222222222222, 'mrr': 0.6477777777777779}
k=100: {'hit_rate': 0.8722222222222222, 'mrr': 0.6477777777777779}
k=200: {'hit_rate': 0.8722222222222222, 'mrr': 0.6477777777777779}
